# QuoridorAI — Stage 6B-1: GPU verification, self-play throughput, pilot

Run **in VS Code with a Colab GPU kernel**: `Select Kernel → Colab → New Colab Server → GPU`.

Before the first cell:
1. Local terminal: `npm run colab:bundle` (prints a `bundle_id`).
2. Explorer: right-click `training/dist/quoridor_az_bundle.tar.gz` → **Upload to Colab** (or copy it to Drive `MyDrive/QuoridorAI/bundles/`).
3. Paste the id into `EXPECTED_BUNDLE_ID` (cell 2).

| Section | What it does | Time on a T4 (rough) |
|---|---|---|
| 1–3 | environment, code bundle, full test suite (incl. Python ⇄ TS fixtures) | ~2 min |
| 4 | Google Drive mount (results and checkpoints go there) | — |
| 5 | GPU verification report → Drive `stage6b1/verification/` | ~10 min |
| 6 | recommended self-play / training settings from the measurements | seconds |
| 7 | training-stability check (2 short iterations) | a few min |
| 8 | **pilot training** — only after 5 succeeded, and only with `RUN_PILOT = True` | ~10–30 min |
| 9 | resume after a runtime reconnect | — |
| 10 | results: before/after, GPU numbers, 6B-2 resource estimate → `stage6b1_summary.json` | seconds |

Nothing here starts a long training run. Every number printed is measured on the device shown next to it;
a run without a GPU is labelled `cpu` and never counts as GPU verification.

## 1. Environment

In [ ]:
import importlib, json, os, platform, subprocess, sys, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
print('Python', platform.python_version(), '|', platform.platform(), '|', os.cpu_count(), 'CPUs')
print('Running on', 'Colab runtime' if IN_COLAB else 'local kernel')
import numpy as np
import torch
print('numpy', np.__version__, '| torch', torch.__version__, '| CUDA build', torch.version.cuda, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU', props.name, f'| {props.total_memory / 2**30:.1f} GiB | compute capability {props.major}.{props.minor} | cuDNN', torch.backends.cudnn.version())
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,memory.used', '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('*** No GPU in this kernel. In VS Code: Select Kernel → Colab → New Colab Server → GPU. ***')
    print('*** Without a GPU the measurements below are CPU numbers and will be labelled as such. ***')

## 2. Project code (bundle → verified install)

In [ ]:
EXPECTED_BUNDLE_ID = ''  # paste the id printed by `npm run colab:bundle`
DRIVE_BUNDLE = Path('/content/drive/MyDrive/QuoridorAI/bundles/quoridor_az_bundle.tar.gz')

import hashlib, tarfile

def _verify(root):
    m = json.loads((root / 'BUNDLE_MANIFEST.json').read_text(encoding='utf-8'))
    bad = [f['path'] for f in m['files'] if hashlib.sha256((root / f['path']).read_bytes()).hexdigest() != f['sha256']]
    if bad:
        raise RuntimeError(f'files differ from manifest: {bad[:5]}')
    return m

if IN_COLAB:
    candidates = [Path('/content/quoridor_az_bundle.tar.gz'), DRIVE_BUNDLE]
    archive = next((p for p in candidates if p.exists()), None)
    if archive is None:
        raise FileNotFoundError('Upload training/dist/quoridor_az_bundle.tar.gz with "Upload to Colab" (or copy it to ' + str(DRIVE_BUNDLE) + ')')
    SRC = Path('/content/quoridor_src')
    if SRC.exists():
        import shutil
        shutil.rmtree(SRC)
    with tarfile.open(archive, 'r:gz') as tar:
        assert all(m.name.startswith('quoridor_src/') and '..' not in m.name for m in tar.getmembers())
        try:
            tar.extractall('/content', filter='data')
        except TypeError:
            tar.extractall('/content')
    manifest = _verify(SRC)
    print('bundle', manifest['bundle_id'], 'from', archive, '| git', (manifest['git_commit'] or 'n/a')[:7], '| uncommitted changes included' if manifest['git_dirty_training_files'] else '')
    if EXPECTED_BUNDLE_ID and manifest['bundle_id'] != EXPECTED_BUNDLE_ID:
        raise RuntimeError(f"bundle {manifest['bundle_id']} != expected {EXPECTED_BUNDLE_ID}: upload the current bundle")
    if importlib.util.find_spec('pytest') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
else:
    SRC = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
    manifest = {'bundle_id': None, 'git_commit': None}
    print('using local source tree', SRC)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
for name in [m for m in sys.modules if m == 'quoridor_az' or m.startswith('quoridor_az.')]:
    del sys.modules[name]
import quoridor_az
from quoridor_az import engine as E, features as F
print('quoridor_az', quoridor_az.__version__, 'from', Path(quoridor_az.__file__).parent)

## 3. Full test suite on this runtime

Cross-language fixtures (433 TS-generated states), fast paths vs the Stage 6A reference implementation (bit-for-bit),
MCTS / virtual loss, self-play, training, checkpoints, failure recovery. The fast engine must not change the rules here either.

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests'], cwd=SRC, capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])
assert r.returncode == 0, 'tests failed — do not continue'

## 4. Google Drive

If `drive.mount` cannot open the sign-in from VS Code, use the command palette action **Colab: Mount Google Drive to Server…** and re-run.

In [ ]:
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        RESULTS_ROOT = Path('/content/drive/MyDrive/QuoridorAI/stage6b1')
    except Exception as e:  # noqa: BLE001
        print('Drive not mounted (', e, ') — results go to /content/stage6b1 and are LOST when the runtime ends')
        RESULTS_ROOT = Path('/content/stage6b1')
else:
    RESULTS_ROOT = SRC / 'runs' / 'stage6b1'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('results ->', RESULTS_ROOT)

## 5. GPU verification

Environment, forward/backward (incl. agreement with the CPU), inference throughput per batch size (fp32, fp16, bf16),
training-step time per batch size, CPU cost of rules/encoding, MCTS speed, a **progressive** self-play sweep
(2 → 8 → 16 → 32 parallel games, 1 and 4 leaves per game; each point runs for `SELFPLAY_SECONDS`) and a checkpoint
round-trip on Drive. Each step records `ok` or its error. The report is saved as JSON in `RESULTS_ROOT/verification`.

In [ ]:
from IPython.display import Markdown, display
from quoridor_az.network import NetConfig, pick_device
from quoridor_az.report import markdown_summary
from quoridor_az.verify import load_latest, run_verification

DEVICE = pick_device()
NET_CONFIG = NetConfig(channels=64, blocks=4)
QUICK = False  # True: fewer batch sizes, 20 s per self-play point
SELFPLAY_SECONDS = 45
PARALLEL = (2, 8, 16, 32)
LEAVES = (1, 4)

gpu_report = run_verification(RESULTS_ROOT / 'verification', DEVICE, NET_CONFIG, quick=QUICK, parallel=PARALLEL, leaves=LEAVES, selfplay_seconds=SELFPLAY_SECONDS)
display(Markdown(markdown_summary(gpu_report)))

## 6. Recommended settings (from the measurements above)

In [ ]:
import matplotlib.pyplot as plt
from quoridor_az.report import recommend_selfplay, recommend_train_batch

gpu_report = gpu_report if 'gpu_report' in globals() else load_latest(RESULTS_ROOT / 'verification', DEVICE.type)
mem_total = gpu_report.get('environment', {}).get('gpu_memory_total_mb')
SP_PICK = recommend_selfplay(gpu_report.get('selfplay'), mem_total)
TB_PICK = recommend_train_batch(gpu_report.get('train_step'))
print('self-play :', SP_PICK)
print('train     :', TB_PICK)

rows = [r for r in gpu_report.get('selfplay') or [] if 'error' not in r]
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for k in sorted({r['leaves_per_game'] for r in rows}):
    rr = [r for r in rows if r['leaves_per_game'] == k]
    ax[0].plot([r['parallel_games'] for r in rr], [r['simulations_per_second'] for r in rr], 'o-', label=f'{k} leaves/game')
    ax[1].plot([r['parallel_games'] for r in rr], [r['mean_batch'] or 0 for r in rr], 'o-', label=f'{k} leaves/game')
ax[0].set_title('self-play simulations / s')
ax[1].set_title('mean network batch')
for a in ax[:2]:
    a.set_xlabel('parallel games')
    a.set_xscale('log', base=2)
    a.legend()
parts = ['select', 'encode', 'legal', 'inference', 'post', 'backup', 'moves_and_samples']
labels = [f"{r['parallel_games']}×{r['leaves_per_game']}" for r in rows]
bottom = np.zeros(len(rows))
for p in parts:
    vals = np.array([r['time_split'][p] for r in rows])
    ax[2].bar(labels, vals, bottom=bottom, label=p)
    bottom += vals
ax[2].set_title('where self-play time goes')
ax[2].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()
for r in rows:
    print(f"{r['parallel_games']:3d}×{r['leaves_per_game']}: CPU side {r.get('cpu_side_us_per_simulation', 0):.0f} µs/sim -> limit {r.get('cpu_side_limit_simulations_per_second') or 0:.0f} sims/s | GPU util {r['utilisation'].get('gpu_util_mean', '-')} % | process CPU {r['utilisation']['process_cpu_utilisation']:.2f} cores")

## 7. Training-stability check (2 short iterations)

Checks before any longer run: policy / value loss can decrease, no NaN/Inf, gradient norms, replay buffer, self-play
win/halt rates and data yield (incl. the halted-game policy), GPU memory per iteration, checkpoint save + restore into a
fresh trainer. Runs in its own directory (`RESULTS_ROOT/stability`, deleted first).

In [ ]:
import shutil
from quoridor_az.pilot import pilot_config, run_pilot

STABILITY_DIR = RESULTS_ROOT / 'stability'
shutil.rmtree(STABILITY_DIR, ignore_errors=True)
par = SP_PICK['parallel_games'] if SP_PICK else 8
leaves = SP_PICK['leaves_per_game'] if SP_PICK else 1
STABILITY_CONFIG = pilot_config(parallel_games=min(par, 4), leaves_per_game=leaves, batch_size=128, games_per_iteration=4, simulations=16, steps_per_iteration=20)
stability = run_pilot(STABILITY_DIR, STABILITY_CONFIG, iterations=2, device=DEVICE, verification=gpu_report, require_gpu=DEVICE.type == 'cuda')
for it in stability['iterations']:
    print({k: it[k] for k in ('iteration', 'games', 'wins', 'halted', 'samples_kept', 'samples_value_masked', 'policy_loss', 'value_loss', 'grad_norm_max', 'non_finite_steps')}, '| memory', it['memory'])
print('learning:', stability['checks']['learning'])
print('resume  :', stability['checks']['resume'])
print('memory  :', stability['checks']['memory'])
print('OK' if stability['ok'] else 'CHECK FAILED — look at the values above')

## 8. Pilot training (small, validates the pipeline — not a strong player)

Runs **only** when `RUN_PILOT = True` and the GPU verification in section 5 succeeded (`gpu_verified`).
Defaults: ResNet 64×4, 8 games per iteration, 32 simulations, the recommended parallel games / leaves and training batch,
40 training steps per iteration, 3 iterations, `halted_policy='policy-only'`, checkpoints on Drive (`RESULTS_ROOT/pilot`).
Re-running the cell continues from the last checkpoint.

In [ ]:
RUN_PILOT = False
PILOT_ITERATIONS = 3
PILOT_DIR = RESULTS_ROOT / 'pilot'
PILOT_CONFIG = pilot_config(
    parallel_games=SP_PICK['parallel_games'] if SP_PICK else 8,
    leaves_per_game=SP_PICK['leaves_per_game'] if SP_PICK else 1,
    batch_size=TB_PICK['batch_size'] if TB_PICK else 256,
    games_per_iteration=8,
    simulations=32,
    steps_per_iteration=40,
    halted_policy='policy-only',
)
print(json.dumps(PILOT_CONFIG.to_dict(), indent=1)[:1500])
pilot = None
if RUN_PILOT:
    pilot = run_pilot(PILOT_DIR, PILOT_CONFIG, PILOT_ITERATIONS, DEVICE, verification=gpu_report)
    print('pilot ok:', pilot['ok'])
else:
    print('RUN_PILOT is False — nothing started')

## 9. Resume after a runtime reconnect

After a disconnect or `Restart`: run cells **1, 2, 4**, then this one. It restores the newest valid checkpoint from Drive,
reports what was restored (and any repairs, e.g. metrics rows of an iteration that never reached a checkpoint), and with
`CONTINUE = True` runs one more pilot iteration from there.

In [ ]:
from quoridor_az.network import pick_device
from quoridor_az.pilot import pilot_config, run_pilot
from quoridor_az.pipeline import Trainer

DEVICE = pick_device()
PILOT_DIR = RESULTS_ROOT / 'pilot'
CONTINUE = False
restored = Trainer(PILOT_DIR, pilot_config(), DEVICE)
if restored.resume():
    print('restored', restored.summary(), '| metrics rows', len(restored.ckpt.read_metrics()), '| warnings', restored.resume_warnings)
    if CONTINUE:
        gpu_report = load_latest(RESULTS_ROOT / 'verification', 'cuda')
        pilot = run_pilot(PILOT_DIR, restored.config, restored.iteration + 1, DEVICE, verification=gpu_report)
        print('resumed run ok:', pilot['ok'], '| resumed flag', pilot['resumed'])
else:
    print('no checkpoint in', PILOT_DIR)

## 10. Results: before/after, GPU numbers, Stage 6B-2 estimate

In [ ]:
from quoridor_az.report import estimate_resources

ab_path = SRC / 'results' / 'stage6b1' / 'ab_compare.json'
if ab_path.exists():
    ab = json.loads(ab_path.read_text(encoding='utf-8'))
    print('CPU before/after (measured locally,', ab['created'], ab['machine']['platform'], ')')
    for row in ab['table']:
        print(f"  {row['metric']:58s} {row['baseline']:10.1f} -> {row['candidate']:10.1f}  ({row['speedup']:.1f}x)")
    print('  identical results:', ab['identical_results'])

gpu_report = load_latest(RESULTS_ROOT / 'verification', 'cuda') or gpu_report
pilot_path = RESULTS_ROOT / 'pilot' / 'pilot_report.json'
pilot = json.loads(pilot_path.read_text(encoding='utf-8')) if pilot_path.exists() else None
summary = {'measured_on': gpu_report.get('measured_on'), 'gpu_verified': gpu_report.get('gpu_verified'), 'gpu': gpu_report.get('environment', {}).get('gpu_name'), 'bundle_id': manifest.get('bundle_id'), 'git_commit': manifest.get('git_commit')}
summary['selfplay_pick'] = recommend_selfplay(gpu_report.get('selfplay'), gpu_report.get('environment', {}).get('gpu_memory_total_mb'))
summary['train_pick'] = recommend_train_batch(gpu_report.get('train_step'))
summary['checkpoint'] = gpu_report.get('checkpoint')
if pilot:
    its = pilot['iterations']
    plies = [it['avg_plies'] for it in its if it.get('avg_plies')]
    summary['pilot'] = {'ok': pilot['ok'], 'iterations': len(its), 'device': pilot['device'], 'avg_plies_per_game': float(np.mean(plies)), 'per_iteration': its, 'checks': pilot['checks']}
    for it in its:
        print(f"pilot iter {it['iteration']}: {it['simulations_per_second']:.0f} sims/s, mean batch {it['mean_batch'] or 0:.1f}, {it['samples_per_second']:.1f} samples/s, self-play {it['selfplay_seconds']:.0f}s, train {it['train_seconds']:.0f}s, loss {it['loss']}, ckpt {it['checkpoint'].get('bytes', 0) / 1e6:.1f} MB")

# Stage 6B-2 candidate plans; the estimate uses measured throughput only (avg plies from the pilot if it ran).
sp = summary['selfplay_pick']
tb = summary['train_pick']
halted_path = SRC / 'results' / 'stage6b1' / 'halted_analysis.json'
if summary.get('pilot', {}).get('avg_plies_per_game'):
    avg_plies = summary['pilot']['avg_plies_per_game']
    plies_source = 'pilot (this GPU run)'
else:
    first = json.loads(halted_path.read_text(encoding='utf-8'))['results'][0]['report']
    avg_plies = first['positions'] / first['games']
    plies_source = 'untrained network, local CPU analysis (halted_analysis.json)'
PLANS = {
    'A (100 iterations × 64 games, 64 sims)': dict(iterations=100, games_per_iteration=64, simulations=64, avg_plies=avg_plies, train_steps_per_iteration=200, keep_checkpoints=5, replay_capacity=200_000),
    'B (100 iterations × 128 games, 128 sims)': dict(iterations=100, games_per_iteration=128, simulations=128, avg_plies=avg_plies, train_steps_per_iteration=400, keep_checkpoints=5, replay_capacity=300_000),
}
if sp and tb:
    sims_s = sp['simulations_per_second']
    summary['estimates'] = {}
    for name, plan in PLANS.items():
        est = estimate_resources(sims_s, tb['seconds_per_step'], plan, checkpoint_bytes=(summary['checkpoint'] or {}).get('checkpoint_bytes'))
        summary['estimates'][name] = est
        print(f"{name}: self-play {est['selfplay_hours']:.1f} h + training {est['train_hours']:.1f} h -> ~{est['total_hours_with_eval']:.1f} h incl. evaluation | replay RAM {est['replay_buffer_memory_gb']:.2f} GB | checkpoints {est.get('checkpoints_gb', 0):.2f} GB")
    print('(throughput measured on', summary['measured_on'], f'; avg plies {avg_plies:.0f} from', plies_source, ')')
out = RESULTS_ROOT / 'stage6b1_summary.json'
out.write_text(json.dumps(summary, indent=1, default=str), encoding='utf-8')
print('saved', out)